# Project 80 — Local Fine-Tuning Evals Harness
## Multi-Task Evaluation Framework for Model Assessment

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Define Eval Suite

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd, time, json

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

eval_suite = {
    "classification": {
        "prompt": "Classify as positive/negative/neutral: {text}\nLabel:",
        "cases": [
            {"text": "Amazing product!", "expected": "positive"},
            {"text": "Terrible service.", "expected": "negative"},
            {"text": "It works fine.", "expected": "neutral"},
            {"text": "Love it!", "expected": "positive"},
            {"text": "Total waste.", "expected": "negative"},
        ],
        "scorer": lambda exp, pred: 1.0 if exp.lower() in pred.lower() else 0.0,
    },
    "extraction": {
        "prompt": "Extract the person's name and date from: {text}\nJSON:",
        "cases": [
            {"text": "Meeting with Alice on Jan 15", "expected": ["alice", "jan"]},
            {"text": "Call from Dr. Bob Chen, March 3", "expected": ["bob", "march"]},
            {"text": "Lunch with Carol — February 28", "expected": ["carol", "february"]},
        ],
        "scorer": lambda exp, pred: sum(1 for e in exp if e in pred.lower()) / len(exp),
    },
    "summarization": {
        "prompt": "Summarize in one sentence: {text}",
        "cases": [
            {"text": "Machine learning models learn from data. They find patterns. "
             "They generalize to new inputs. Overfitting is a common problem.",
             "expected": ["learn", "data", "pattern"]},
            {"text": "Python is a programming language. It's easy to read. "
             "It supports multiple paradigms. It has a large ecosystem.",
             "expected": ["python", "language"]},
        ],
        "scorer": lambda exp, pred: sum(1 for w in exp if w in pred.lower()) / len(exp),
    },
    "reasoning": {
        "prompt": "Answer: {text}",
        "cases": [
            {"text": "If all roses are flowers and all flowers need water, do roses need water?",
             "expected": ["yes"]},
            {"text": "A bat and ball cost $1.10. The bat costs $1 more than the ball. "
             "What does the ball cost?",
             "expected": ["0.05", "5 cent", "five cent"]},
        ],
        "scorer": lambda exp, pred: 1.0 if any(e in pred.lower() for e in exp) else 0.0,
    },
}
print(f"Eval suite: {len(eval_suite)} tasks, "
      f"{sum(len(t['cases']) for t in eval_suite.values())} total cases")

## Step 2 — Run Full Evaluation

In [ ]:
results = []
for task_name, task_config in eval_suite.items():
    prompt = ChatPromptTemplate.from_template(task_config["prompt"])
    chain = prompt | llm | StrOutputParser()
    scorer = task_config["scorer"]

    for case in task_config["cases"]:
        start = time.time()
        output = chain.invoke({"text": case["text"]})
        elapsed = time.time() - start
        score = scorer(case["expected"], output)

        results.append({
            "task": task_name,
            "input": case["text"][:40],
            "score": round(score, 2),
            "latency_s": round(elapsed, 2),
            "output_len": len(output),
        })

df = pd.DataFrame(results)
print("EVALUATION RESULTS")
print("=" * 60)
print(df.to_string(index=False))

## Step 3 — Task-Level Analysis

In [ ]:
print("TASK-LEVEL SUMMARY")
print("=" * 50)
task_summary = df.groupby("task").agg({
    "score": ["mean", "min", "max", "std"],
    "latency_s": "mean",
}).round(3)
print(task_summary.to_string())

print(f"\nOVERALL METRICS:")
print(f"  Mean score:    {df['score'].mean():.0%}")
print(f"  Median score:  {df['score'].median():.0%}")
print(f"  Mean latency:  {df['latency_s'].mean():.2f}s")

# Weakest area
weakest = df.groupby("task")["score"].mean().idxmin()
print(f"\nWeakest task: {weakest} ({df.groupby('task')['score'].mean()[weakest]:.0%})")
print(f"Strongest task: {df.groupby('task')['score'].mean().idxmax()} ({df.groupby('task')['score'].mean().max():.0%})")

## Step 4 — Pass/Fail Report Card

In [ ]:
thresholds = {"classification": 0.8, "extraction": 0.7, "summarization": 0.6, "reasoning": 0.5}

print("REPORT CARD")
print("=" * 50)
all_pass = True
for task, threshold in thresholds.items():
    avg = df[df["task"] == task]["score"].mean()
    passed = avg >= threshold
    all_pass = all_pass and passed
    icon = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {icon} {task:<20} {avg:.0%} (threshold={threshold:.0%})")

print(f"\nOverall: {'✓ ALL PASSED' if all_pass else '✗ NEEDS IMPROVEMENT'}")

if not all_pass:
    print("\nFailed tasks need attention before fine-tuning is worthwhile.")
    print("Consider improving prompt strategy or collecting more training data.")

## What You Learned
- **Multi-task evaluation harness** with custom scorers
- **Task-specific thresholds** for pass/fail gates
- **Performance profiling** per task category
- **Report card generation** for model readiness assessment